In [75]:
# Mount Google Drive to access dataset files stored in Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [76]:
# Import core libraries for numerical operations (numpy) and data manipulation (pandas)
import numpy as np
import pandas as pd

In [77]:
# Load raw dataset from Google Drive into a pandas DataFrame
# Preview first few rows to verify successful loading and structure

csv_file = '/content/drive/My Drive/EADA/EADA_DeepLearning/Project/google_books_dataset.csv'

df = pd.read_csv(csv_file)

df.head()

,book_id,title,subtitle,authors,publisher,published_date,description,page_count,categories,average_rating,...,language,preview_link,info_link,isbn_13,isbn_10,list_price,currency,buyable,search_category,thumbnail
0,LR_VDQAAQBAJ,Bestsellers,"The Path (bestsellers, free bestsellers, bests...","Ivan King, bestsellers",bestsellers,2017-01-04,"Hear What the Critics are Saying ""Wow, what an...",70.0,Young Adult Fiction,NaN,...,en,http://books.google.com/books?id=LR_VDQAAQBAJ&...,https://play.google.com/store/books/details?id...,NaN,NaN,12.99,USD,True,fiction bestsellers,http://books.google.com/books/content?id=LR_VD...
1,WcjTDQAAQBAJ,Bestsellers,"Hell: A Place Without Hope (bestseller books, ...","Ivan King, bestsellers",bestsellers,2017-01-03,"Hear What the Critics are Saying Wow, very ins...",32.0,Comics & Graphic Novels,NaN,...,en,http://books.google.com/books?id=WcjTDQAAQBAJ&...,https://play.google.com/store/books/details?id...,NaN,NaN,9.99,USD,True,fiction bestsellers,http://books.google.com/books/content?id=WcjTD...
2,4fXUDAAAQBAJ,The Bestseller Code,Anatomy of the Blockbuster Novel,"Jodie Archer, Matthew L. Jockers",Macmillan,2016-09-20,"""What if there was an algorithm that could pre...",253.0,Business & Economics,NaN,...,en,http://books.google.com/books?id=4fXUDAAAQBAJ&...,http://books.google.com/books?id=4fXUDAAAQBAJ&...,9.781250e+12,1250088275,NaN,NaN,False,fiction bestsellers,http://books.google.com/books/content?id=4fXUD...
3,yIVuDwAAQBAJ,Bestseller,A Century of America's Favorite Books,Robert McParland,Bloomsbury Publishing PLC,2018-12-15,Whether curled up on a sofa with a good myster...,335.0,Literary Criticism,NaN,...,en,http://books.google.com/books?id=yIVuDwAAQBAJ&...,https://play.google.com/store/books/details?id...,9.781538e+12,1538110008,40.50,USD,True,fiction bestsellers,http://books.google.com/books/content?id=yIVuD...
4,2JHXwAEACAAJ,Bestsellers: Popular Fiction since 1900,NaN,C. Bloom,Palgrave Macmillan,2002-07-09,This guide and reference work of all of the be...,306.0,Literary Criticism,NaN,...,en,http://books.google.com/books?id=2JHXwAEACAAJ&...,http://books.google.com/books?id=2JHXwAEACAAJ&...,9.780334e+12,0333687426,NaN,NaN,False,fiction bestsellers,http://books.google.com/books/content?id=2JHXw...


In [78]:
# Standardize data types:
# - Convert published_date to datetime (invalid values → NaT)
# - Convert isbn_13 to string to preserve full identifier (avoid scientific notation issues)
# - Ensure page_count is integer, replacing missing values with 0
df['published_date'] = pd.to_datetime(df['published_date'], errors='coerce')
df['isbn_13'] = df['isbn_13'].fillna(0).astype('int64').astype(str)
df['page_count'] = df['page_count'].fillna(0).astype(int)

In [79]:
# Remove records without titles (critical identifier)
# Fill missing authors and publisher with placeholder to maintain structural consistency
# Preserve NaNs in analytical fields (e.g., ratings) for later modeling
df = df.dropna(subset=['title'])
df['authors'] = df['authors'].fillna('Unknown')
df['publisher'] = df['publisher'].fillna('Unknown')


In [80]:
# Standardize text fields:
# - Convert to string type
# - Remove leading/trailing spaces
# - Convert all text to lowercase for consistency and deduplication
text_cols = ['title','subtitle','authors','publisher','categories']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

In [81]:
# Clean author field:
# - Remove non-author keywords like "bestseller(s)"
# - Fix malformed commas and spacing in author lists
df['authors'] = df['authors'].str.replace(r'\bbestsellers?\b', '', regex=True)
df['authors'] = df['authors'].str.replace(r'\s+,', ',', regex=True)

In [82]:
# Create normalized title key by removing special characters
# Remove duplicate books based on cleaned title + publication date
# This reduces redundancy caused by formatting inconsistencies
df['title_clean'] = df['title'].str.replace(r'[^a-z0-9]', '', regex=True)
df = df.drop_duplicates(subset=['title_clean','published_date'])

In [83]:
# Filter out invalid records:
# - Remove books with zero or negative page count
# - Ensure ratings are within valid range (≤ 5), keep NaNs for missing values
df = df[df['page_count'] > 0]
df = df[(df['average_rating'].isna()) | (df['average_rating'] <= 5)]

In [84]:
# Standardize categorical fields:
# - Language → lowercase for consistency
# - Currency → uppercase (ISO-like format)
df['language'] = df['language'].str.lower()
df['currency'] = df['currency'].str.upper()

In [57]:
# Save cleaned csv
# df.to_csv('/content/drive/My Drive/EADA/EADA_DeepLearning/Project/cleaned_books_dataset.csv', index=False)

In [85]:
# Mood mapping
# Define heuristic keyword mapping to later infer emotional tone (mood) from book descriptions
# This is NOT yet applied — only defines the mapping for future feature engineering
mood_map = {
    'dark': ['death','war','murder','crime'],
    'inspiring': ['success','growth','achievement'],
    'romantic': ['love','relationship','heart'],
    'adventurous': ['journey','quest','explore']
}

In [86]:
# Create derived features:
# - reading_time: estimated reading duration (hours)
# - length_category: discretized book size (short/medium/long)
df['reading_time'] = df['page_count'] / 30  # rough hours
df['length_category'] = pd.cut(df['page_count'],
                              bins=[0,150,400,1000],
                              labels=['short','medium','long'])

In [89]:
# Retrieve book description from Google Books API and validate using title similarity
import requests

def get_description_google(isbn, title):
    url = f"https://www.googleapis.com/books/v1/volumes?q=isbn:{isbn}"
    r = requests.get(url).json()
    try:
        item = r['items'][0]['volumeInfo']
        if is_match(title, item.get('title','')):
            return item.get('description', None)
        else:
            return None
    except:
        return None

In [90]:
# Fill missing descriptions using external API (Google Books) based on ISBN + title
df['description'] = df.apply(
    lambda row: get_description_google(row['isbn_13'],row['title'])
    if pd.isna(row['description']) else row['description'],
    axis=1
)

In [91]:
# Define keyword dictionary to infer book categories based on description content
category_keywords = {
    'fiction': ['novel','story','fiction'],
    'science': ['physics','biology','research'],
    'business': ['finance','market','startup'],
    'self-help': ['success','mindset','growth'],
    'history': ['war','ancient','empire']
}

In [92]:
# Infer multiple categories from description using keyword frequency threshold
# Returns list of matched categories or empty list if no strong signal

def infer_category_multi(text):
    text = str(text).lower()
    result = []
    for cat, words in category_keywords.items():
        score = sum(w in text for w in words)
        if score >= 2:  # threshold
            result.append(cat)
    return result if result else []

In [93]:
# Define missing-value checker FIRST
def is_empty(x):
    return (x is None) or (x == []) or (str(x).lower() == 'nan')

# Fetch categories from API
def get_categories_google(isbn):
    url = f"https://www.googleapis.com/books/v1/volumes?q=isbn:{isbn}"
    try:
        r = requests.get(url).json()
        return r['items'][0]['volumeInfo'].get('categories', None)
    except:
        return None

# Apply API fill
df['categories'] = df.apply(
    lambda row: get_categories_google(row['isbn_13'])
    if is_empty(row['categories'])
    else row['categories'],
    axis=1
)

In [96]:
# Convert comma-separated authors and '&'-separated categories into list format
df['authors'] = df['authors'].apply(
    lambda x: [a.strip() for a in x.split(',')] if isinstance(x, str) else x
)

df['categories'] = df['categories'].apply(
    lambda x: [c.strip() for c in x.split('&')] if isinstance(x, str) else x
)

In [100]:
# Ensure categories and authors are always stored as lists for consistency
def ensure_list(x):
    if isinstance(x, list):
        return x
    elif pd.isna(x) or x is None:
        return []
    else:
        return [x]

df['categories'] = df['categories'].apply(ensure_list)

df['authors'] = df['authors'].apply(ensure_list)

In [101]:
# Analyze category distribution to detect imbalance or bias in dataset
# Used for validation AFTER all category filling is complete
cat_dist = df['categories'].explode().value_counts(normalize=True)
print(cat_dist.head(10))

categories
fiction               0.071789
computers             0.056307
economics             0.044003
business              0.044003
history               0.027379
education             0.026076
cooking               0.021023
medical               0.020535
literary criticism    0.019231
biography             0.018334
Name: proportion, dtype: float64


In [102]:
# Create a unified text field
df['combined_text'] = (
    df['title'] + ' ' +
    df['subtitle'].fillna('') + ' ' +
    df['authors'].apply(lambda x: ' '.join(x)) + ' ' +
    df['categories'].apply(lambda x: ' '.join(x)) + ' ' +
    df['description'].fillna('')
)

In [103]:
# Provide fallback category inference using title keywords
# Used only when API and description inference fail
def infer_from_title(title):
    t = str(title).lower()
    if 'guide' in t or 'how to' in t:
        return 'self-help'
    if 'history' in t:
        return 'history'
    if 'science' in t:
        return 'science'
    return None

In [104]:
# Hierarchical category resolution pipeline:
# 1. API-based enrichment (highest confidence)
# 2. Description-based inference (medium confidence)
# 3. Title-based fallback (lowest confidence)
# Tracks provenance of category assignment via 'category_source'
df['category_source'] = 'original'

# API fill
mask_api = df['categories'].apply(is_empty)
df.loc[mask_api, 'categories'] = df.loc[mask_api].apply(
    lambda row: get_categories_google(row['isbn_13']), axis=1
)
df.loc[mask_api, 'category_source'] = 'api'

# Description inference
mask_desc = df['categories'].apply(is_empty)
df.loc[mask_desc, 'categories'] = df.loc[mask_desc].apply(
    lambda row: infer_category_multi(row['description']), axis=1
)
df.loc[mask_desc, 'category_source'] = 'desc_inferred'

# Title fallback
mask_title = df['categories'].apply(is_empty)
df.loc[mask_title, 'categories'] = df.loc[mask_title].apply(
    lambda row: [infer_from_title(row['title'])] if infer_from_title(row['title']) else [], axis=1
)
df.loc[mask_title, 'category_source'] = 'title_inferred'

In [105]:
# Standardize text for authors and categories:
# - remove punctuation and normalize casing
# - ensure consistent representation for embedding and matching
# - build frequency-based canonical author mapping (placeholder for clustering)
import re
from collections import Counter

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

# Normalize authors
df['authors'] = df['authors'].apply(lambda x: [normalize_text(a) for a in x])

# Build canonical mapping (most frequent form wins)
all_authors = [a for sublist in df['authors'] for a in sublist]
freq = Counter(all_authors)

canonical_map = {a: a for a in freq}  # can improve with clustering later

df['authors'] = df['authors'].apply(lambda x: [canonical_map[a] for a in x])

# Normalize categories
df['categories'] = df['categories'].apply(lambda x: [normalize_text(c) for c in x])

In [106]:
# Define fuzzy matching function to validate API results (title similarity check)
from difflib import SequenceMatcher

def is_match(title1, title2):
    return SequenceMatcher(None, title1, title2).ratio() > 0.7

In [107]:
# Save cleaned csv
df.to_csv('/content/drive/My Drive/EADA/EADA_DeepLearning/Project/cleanver_books_dataset.csv', index=False)